# Ddri 최종 지구판단 군집화 피처 생성 노트북

목적:
- 내일 발표용 2차 군집화에서 바로 검토할 수 있는 `지구판단 중심 피처 세트`를 만든다.
- 기존 1차 군집화 active station 범위에 맞춰 train/test용 최종 피처 CSV를 생성한다.

이번 노트북에서 묶는 피처:
- 도착 시간대 비율 3종
- 시간대별 순유입 2종
- 교통 접근성 2종
- 생활인구 시간대 평균 3종

출력:
- `cheng80/ddri_final_district_clustering_features_train_2023_2024.csv`
- `cheng80/ddri_final_district_clustering_features_test_2025.csv`

## 설계 원칙

- 1차 군집화와 동일한 station 범위로 비교하기 위해 active station 목록을 그대로 사용한다.
- 도착 시간대와 순유입은 지구 성격 해석의 주력 피처로 둔다.
- 교통 접근성과 생활인구는 지구판단 보강용으로 함께 붙인다.
- 결과는 기존 군집화 CSV와 합치지 않고, 별도 검토용 파일로 저장한다.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
BASELINE_DIR = BASE_DIR / 'works' / '01_clustering' / '01_baseline'
RAW_DIR = BASE_DIR / '3조 공유폴더'
CHENG80_DIR = BASE_DIR / 'cheng80'
DATA_DIR = BASE_DIR / 'works' / '01_clustering' / '06_data'

if str(BASELINE_DIR) not in sys.path:
    sys.path.append(str(BASELINE_DIR))

from ddri_station_clustering_baseline import file_groups, load_clean_group, load_station_frames

CHENG80_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR

PosixPath('/Users/cheng80/Desktop/ddri_work')

## active station 기준 로드

train/test에 실제로 feature가 생성된 대여소만 이번 최종 세트의 행 대상으로 쓴다.
이렇게 해야 기존 1차 군집화 결과와 바로 비교할 수 있다.

In [2]:
train_base = pd.read_csv(DATA_DIR / 'ddri_station_cluster_features_train_2023_2024.csv')
test_base = pd.read_csv(DATA_DIR / 'ddri_station_cluster_features_test_2025.csv')
train_station_ids = sorted(train_base['station_id'].tolist())
test_station_ids = sorted(test_base['station_id'].tolist())

print('train_station_count =', len(train_station_ids))
print('test_station_count =', len(test_station_ids))

train_station_count = 164
test_station_count = 162


## 기존 생성된 지구판단 피처 로드

이전 단계에서 만든 도착 시간대 비율 / 순유입 피처를 불러온다.

In [3]:
train_district = pd.read_csv(CHENG80_DIR / 'ddri_station_cluster_district_features_train_2023_2024.csv')
test_district = pd.read_csv(CHENG80_DIR / 'ddri_station_cluster_district_features_test_2025.csv')

train_district = train_district[train_district['station_id'].isin(train_station_ids)].copy()
test_district = test_district[test_district['station_id'].isin(test_station_ids)].copy()

print(len(train_district), len(test_district))

164 162


## 교통 접근성 피처 생성 함수

기존 환경 스크립트는 train cluster 결과를 기준으로 merge되어 있어, 여기서는 필요한 두 피처만 다시 계산한다.
- `subway_distance_m`
- `bus_stop_count_300m`

In [4]:
def haversine_matrix(lat1, lon1, lat2, lon2):
    r = 6371000.0
    lat1 = np.radians(lat1)[:, None]
    lon1 = np.radians(lon1)[:, None]
    lat2 = np.radians(lat2)[None, :]
    lon2 = np.radians(lon2)[None, :]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return r * c

def build_transport_features(target_station_ids):
    stations = pd.read_csv(DATA_DIR / 'ddri_common_station_master.csv')
    stations = stations[stations['대여소번호'].isin(target_station_ids)].copy()
    stations = stations.rename(columns={'대여소번호': 'station_id', '위도': 'station_lat', '경도': 'station_lon'})
    stations['station_lat'] = pd.to_numeric(stations['station_lat'], errors='coerce')
    stations['station_lon'] = pd.to_numeric(stations['station_lon'], errors='coerce')

    subway = pd.read_csv(
        RAW_DIR / '[교통데이터] 지하철 정보/서울시 역사마스터 정보/서울시 역사마스터 정보.csv',
        encoding='cp949'
    )
    subway = subway.rename(columns={'위도': 'subway_lat', '경도': 'subway_lon'})
    subway['subway_lat'] = pd.to_numeric(subway['subway_lat'], errors='coerce')
    subway['subway_lon'] = pd.to_numeric(subway['subway_lon'], errors='coerce')
    subway = subway.dropna(subset=['subway_lat', 'subway_lon']).copy()

    bus = pd.read_csv(
        RAW_DIR / '서울시 버스정류소 위치정보/2024년/2024년1~4월1일기준_서울시버스정류소위치정보.csv',
        encoding='cp949'
    )
    bus = bus.rename(columns={'CRDNT_Y': 'bus_lat', 'CRDNT_X': 'bus_lon'})
    bus['bus_lat'] = pd.to_numeric(bus['bus_lat'], errors='coerce')
    bus['bus_lon'] = pd.to_numeric(bus['bus_lon'], errors='coerce')
    bus = bus.dropna(subset=['bus_lat', 'bus_lon']).copy()

    subway_dist = haversine_matrix(
        stations['station_lat'].to_numpy(), stations['station_lon'].to_numpy(),
        subway['subway_lat'].to_numpy(), subway['subway_lon'].to_numpy(),
    )
    bus_dist = haversine_matrix(
        stations['station_lat'].to_numpy(), stations['station_lon'].to_numpy(),
        bus['bus_lat'].to_numpy(), bus['bus_lon'].to_numpy(),
    )

    out = stations[['station_id']].copy()
    out['subway_distance_m'] = subway_dist.min(axis=1)
    out['bus_stop_count_300m'] = (bus_dist <= 300).sum(axis=1)
    return out.sort_values('station_id').reset_index(drop=True)


In [5]:
train_transport = build_transport_features(train_station_ids)
test_transport = build_transport_features(test_station_ids)
train_transport.head()

,station_id,subway_distance_m,bus_stop_count_300m
0,2301,676.393220,20
1,2302,129.301580,40
2,2303,31.250222,40
3,2304,455.995586,12
4,2305,683.890247,16


## 생활인구 시간대 피처 생성 함수

생활인구는 `행정동코드`와 `시간대구분` 기준으로 집계한다.

이번 노트북에서는 아래 3개 시간대 평균만 사용한다.
- `07~10시`
- `11~14시`
- `17~20시`

각 대여소는 먼저 `행정동코드`를 붙인 뒤, 해당 동의 시간대 평균 생활인구를 가져온다.

In [6]:
station_dong = pd.read_csv(CHENG80_DIR / 'ddri_station_to_dong_mapping_2025.csv', dtype={'mapped_dong_code': str})
station_dong = station_dong.rename(columns={'대여소번호': 'station_id'})[['station_id', 'mapped_dong_code', 'mapped_dong_name']].copy()

def read_life_population_csv(fp: Path) -> pd.DataFrame:
    read_kwargs = {'dtype': {'행정동코드': str, '기준일ID': str, '시간대구분': str}}
    try:
        return pd.read_csv(fp, encoding='utf-8', index_col=False, **read_kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(fp, encoding='cp949', index_col=False, **read_kwargs)

def load_life_population(year: int) -> pd.DataFrame:
    path = RAW_DIR / '서울특별시행정동별 서울생활인구(내국인) 2025년' / f'{year}년' / f'LOCAL_PEOPLE_DONG_{year}01.csv'
    # 연도별 모든 월 파일을 합친다.
    folder = path.parent
    files = sorted(folder.glob('LOCAL_PEOPLE_DONG_*.csv'))
    frames = []
    for fp in files:
        df = read_life_population_csv(fp)
        df['시간대구분'] = df['시간대구분'].str.zfill(2)
        df['총생활인구수'] = pd.to_numeric(df['총생활인구수'], errors='coerce')
        frames.append(df[['행정동코드', '시간대구분', '총생활인구수']])
    return pd.concat(frames, ignore_index=True)

def summarize_life_population(years):
    pop_df = pd.concat([load_life_population(y) for y in years], ignore_index=True)
    pop_df = pop_df[pop_df['행정동코드'].str.startswith('11680', na=False)].copy()

    def window_mean(hours):
        return (
            pop_df[pop_df['시간대구분'].isin([str(h).zfill(2) for h in hours])]
            .groupby('행정동코드')['총생활인구수']
            .mean()
        )

    morning = window_mean(range(7, 11))
    lunch = window_mean(range(11, 15))
    evening = window_mean(range(17, 21))

    out = pd.DataFrame({'mapped_dong_code': sorted(pop_df['행정동코드'].dropna().unique())})
    out['life_pop_7_10_mean'] = morning.reindex(out['mapped_dong_code']).values
    out['life_pop_11_14_mean'] = lunch.reindex(out['mapped_dong_code']).values
    out['life_pop_17_20_mean'] = evening.reindex(out['mapped_dong_code']).values
    return out


In [7]:
train_life = summarize_life_population([2023, 2024])
test_life = summarize_life_population([2025])
train_life.head()

,mapped_dong_code,life_pop_7_10_mean,life_pop_11_14_mean,life_pop_17_20_mean
0,11680510,34711.262650,47294.581114,44266.389853
1,11680521,43807.053901,51912.863197,45663.383200
2,11680531,48771.596214,62230.211239,48357.276452
3,11680545,40653.442095,53924.654358,48026.726368
4,11680565,44485.181326,53001.089907,42721.223895


## 최종 피처 테이블 결합

station_id 기준 active station 목록에 맞춰,
도착 시간대 + 순유입 + 교통 접근성 + 생활인구를 하나의 검토용 테이블로 결합한다.

In [8]:
def build_final_feature_table(station_ids, district_df, transport_df, life_df):
    base = pd.DataFrame({'station_id': station_ids})
    out = (
        base.merge(district_df, on='station_id', how='left')
        .merge(transport_df, on='station_id', how='left')
        .merge(station_dong, on='station_id', how='left')
        .merge(life_df, on='mapped_dong_code', how='left')
        .sort_values('station_id')
        .reset_index(drop=True)
    )
    return out[
        [
            'station_id',
            'mapped_dong_code',
            'mapped_dong_name',
            'arrival_7_10_ratio',
            'arrival_11_14_ratio',
            'arrival_17_20_ratio',
            'morning_net_inflow',
            'evening_net_inflow',
            'subway_distance_m',
            'bus_stop_count_300m',
            'life_pop_7_10_mean',
            'life_pop_11_14_mean',
            'life_pop_17_20_mean',
        ]
    ]

train_final = build_final_feature_table(train_station_ids, train_district, train_transport, train_life)
test_final = build_final_feature_table(test_station_ids, test_district, test_transport, test_life)

print('train_final_rows =', len(train_final))
print('test_final_rows =', len(test_final))
train_final.head()

train_final_rows = 164
test_final_rows = 162


,station_id,mapped_dong_code,mapped_dong_name,arrival_7_10_ratio,arrival_11_14_ratio,arrival_17_20_ratio,morning_net_inflow,evening_net_inflow,subway_distance_m,bus_stop_count_300m,life_pop_7_10_mean,life_pop_11_14_mean,life_pop_17_20_mean
0,2301,11680510,신사동,0.075694,0.126693,0.373307,7.0,1754.0,676.393220,20,34711.262650,47294.581114,44266.389853
1,2302,11680521,논현1동,0.128852,0.165705,0.380852,-125.0,1922.0,129.301580,40,43807.053901,51912.863197,45663.383200
2,2303,11680521,논현1동,0.117521,0.158425,0.359829,-2241.0,910.0,31.250222,40,43807.053901,51912.863197,45663.383200
3,2304,11680531,논현2동,0.161560,0.183048,0.319936,-751.0,-1352.0,455.995586,12,48771.596214,62230.211239,48357.276452
4,2305,11680531,논현2동,0.429047,0.202975,0.167156,1661.0,-1498.0,683.890247,16,48771.596214,62230.211239,48357.276452


In [9]:
train_final.isna().sum()

station_id             0
mapped_dong_code       0
mapped_dong_name       0
arrival_7_10_ratio     0
arrival_11_14_ratio    0
arrival_17_20_ratio    0
morning_net_inflow     0
evening_net_inflow     0
subway_distance_m      0
bus_stop_count_300m    0
life_pop_7_10_mean     3
life_pop_11_14_mean    3
life_pop_17_20_mean    3
dtype: int64

## CSV 저장

검토용 최종 피처는 `cheng80/`에 저장한다.

In [10]:
train_path = CHENG80_DIR / 'ddri_final_district_clustering_features_train_2023_2024.csv'
test_path = CHENG80_DIR / 'ddri_final_district_clustering_features_test_2025.csv'

train_final.to_csv(train_path, index=False)
test_final.to_csv(test_path, index=False)

print('saved:', train_path)
print('saved:', test_path)

saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_final_district_clustering_features_train_2023_2024.csv
saved: /Users/cheng80/Desktop/ddri_work/cheng80/ddri_final_district_clustering_features_test_2025.csv
